# CKIN2 — others

Lecture du patrimoine consolidé avec les mêmes quantités, prix et couvertures que le dashboard.
Les références d’acquisition restent séparées des observations de marché.


## 1. Périmètre et chargement partagé

`CKIN2_DATA_ROOT` choisit la racine des données et `CKIN2_AS_OF` fixe la date d’analyse.
En mode actuel, cette date borne les prix, pas la date des quantités. Pour une analyse passée,
activer `HISTORICAL` : seules les quantités des snapshots enregistrés sont alors utilisées.
Les imports publics visibles ne remplacent jamais le patrimoine.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from notebook_reporting import french_display, FrenchPlots, portfolio_summary
display = french_display(display)
px = FrenchPlots(px)
from analyze_categories import category_price_figures
from notebook_data import category_report, notebook_context

DATA_ROOT, AS_OF = notebook_context(PROJECT_ROOT)
HISTORICAL = False  # True : dernier snapshot enregistré à AS_OF.
result = category_report(DATA_ROOT, categories=['sticker', 'charm', 'agent'], as_of=AS_OF, historical=HISTORICAL)
prices, acquisitions = result["prices"], result["acquisitions"]
valuation, summary = result["holdings"], result["summary"]
portfolio_summary(result, "Synthèse du patrimoine sélectionné", display)
print("Mode :", "quantités du snapshot enregistré" if HISTORICAL else "quantités actuelles connues")
print("Prix disponibles :", len(prices), "observations ; références d’acquisition affichées séparément.")


## 2. Evolution des prix de marche et acquisitions

Les croix indiquent les valeurs de reference aux dates d acquisition ; elles ne sont pas reliees aux courbes de marche.
La base 100, la couverture et la valorisation utilisent uniquement les releves de marche (source_column > 0).
Les dates estimees sont indiquees au survol des points. Les acquisitions figurent aussi dans une chronologie distincte.


In [ ]:
if not prices.empty or not acquisitions.empty:
    absolute, relative = category_price_figures(prices, acquisitions)
    absolute.show()
    if not prices.empty:
        relative.show()
    else:
        print("Aucun prix de marche disponible : indices et valorisation indisponibles.")
else:
    print("Aucune observation disponible.")
if not acquisitions.empty:
    px.scatter(acquisitions, x="date", y="item_name",
               title="Dates d'acquisition - achats ou drops, sans cout confirme").show()


## 3. Valeur de l'inventaire aux derniers prix connus

Le sous-total connu exclut les quantites ou prix manquants. Le total reste indisponible si la couverture est incomplete.
Les prix d'achat sans date restent utilisables pour comparer le cout et la valeur ; aucun rendement annualise n'est calcule.


In [ ]:
priced = valuation.loc[valuation.market_value.notna() & valuation.quantity.gt(0)]
if not priced.empty:
    px.bar(priced.sort_values("market_value"), x="market_value", y="item_name",
           orientation="h", color="category", title="Contribution au sous-total connu (EUR)").show()


## 4. Historique enregistré et limites des coûts

L’historique utilise les quantités enregistrées à chaque snapshot, sans projection du patrimoine actuel.
Une catégorie non importée reste indisponible ; un sous-total connu n’est pas un total complet.


In [ ]:
purchases = valuation.loc[valuation.purchase_unit_price.notna(),
    ["item_name", "quantity", "purchase_unit_price", "purchase_date", "purchase_cost", "market_value", "unrealized_change"]]
if purchases.empty:
    print("Aucun coût déclaré applicable aux quantités de cette sélection.")
else:
    print("Coûts déclarés dans le CSV ; écarts bruts sans frais, sans preuve du coût FIFO des lots actuels.")
    display(purchases)
print("Les coûts ne sont jamais déduits des références d’acquisition ni transférés à un remplacement propriétaire.")
print("Snapshots enregistrés :", result["historical"].snapshot_date.nunique())


## Annexe — diagnostics et données détaillées

Tables de contrôle et détails conservés pour la vérification des résultats.


In [ ]:
display(summary)
display(valuation)
display(acquisitions)
for group in result["groups"]:
    display(group["coverage"])
display(result["historical"])
display(result["contributions"])
